In [1]:
import os

In [2]:
%pwd

'c:\\Users\\AJAY\\Documents\\ML Projects\\TextSummarizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\AJAY\\Documents\\ML Projects\\TextSummarizer'

### Basic Configuration

In [15]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class ModelTrainingConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs : int
    warmup_steps : int
    per_device_train_batch_size : int
    weight_decay : float
    logging_steps : int
    eval_strategy: str
    eval_steps : int
    save_steps : float
    gradient_accumulation_steps : int

In [16]:
from src.TextSummarizer.constants import *
import unittest

# Compatibility fix for libraries using the removed Python 2/older Python API
if not hasattr(unittest.TestCase, "assertRaisesRegexp"):
    unittest.TestCase.assertRaisesRegexp = unittest.TestCase.assertRaisesRegex

from src.TextSummarizer.utils.common import read_yaml, create_directories

### Configuration update

In [17]:
class ConfigurationManager:
    def __init__(self,
                 config_path = CONFIG_FILE_PATH,
                 params_path =  PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainingConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_traniner_config = ModelTrainingConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            model_ckpt =  config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps  = params.logging_steps,
            eval_strategy = params.eval_strategy,
            eval_steps  = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )
        return model_traniner_config
        

### Components

In [18]:
import os
from src.TextSummarizer.logging import logger


from transformers import AutoModelForSeq2SeqLM,AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
import torch

from datasets import load_from_disk


In [ ]:
class ModelTrainer:
    def __init__(self,config: ModelTrainingConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt) 
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        ## Loading the data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
                    output_dir=self.config.root_dir,
                    num_train_epochs=1,
                    warmup_steps=500,
                    per_device_train_batch_size=1,
                    per_device_eval_batch_size=1,
                    weight_decay=0.01,
                    logging_steps=10,
                    eval_strategy='steps',
                    eval_steps=500,
                    save_steps=1e6,
                    gradient_accumulation_steps=16,
                    dataloader_pin_memory=False,
                    dataloader_num_workers=0
                )

        trainer = Trainer(model=model_pegasus, args=trainer_args,
                  data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["test"],
                  eval_dataset=dataset_samsum_pt["validation"])


        trainer.train()

        ## Save model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsum-model"))

        ## Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))


In [10]:
# !pip install --upgrade accelerate
# !pip uninstall -y transformers accelerate
# !pip install transformers accelerate

In [11]:
config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()
model_trainer = ModelTrainer(config=model_trainer_config)
model_trainer.train()

[2026-09-03 22:05:16,832: INFO: common]: yaml file: config\config.yaml loaded successfully]
[2026-09-03 22:05:16,849: INFO: common]: yaml file: params.yaml loaded successfully]
[2026-09-03 22:05:16,852: INFO: common]: created directory at: artifacts]
[2026-09-03 22:05:16,855: INFO: common]: created directory at: artifacts/model_trainer]


[2026-09-03 22:05:17,679: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]


[2026-09-03 22:05:17,681: WARNING: _http]: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-09-03 22:05:17,693: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-09-03 22:05:17,992: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-09-03 22:05:18,006: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-09-03 22:05:18,255: INFO: _client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 4473.76it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[2026-09-03 22:05:51,351: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-09-03 22:05:51,391: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/generation_config.json "HTTP/1.1 200 OK"]
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\AJAY\Documents\ML Projects\TextSummarizer\env\Lib\site-packages\IPython\core\interactiveshell.py", line 2235, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\AJAY\Documents\ML Projects\TextSummarizer\env\Lib\site-packages\IPython\core\ultratb.py", line 1272, in structured_traceback
    return FormattedTB.structured_traceback(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\AJAY\Documents\ML Projects\TextSummarizer\env\Lib\site-packages\IPython\core\ultratb.py", line 1138, in structured_traceback
    return VerboseTB.structured_traceback(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\AJAY\Documents\ML Projects\TextSummarizer\env\Lib\site-packages\IPython\core\ultratb.py", line 946, in structured_traceback
    formatted_exceptions: list[list[str]] = self.format_exception_as_a_whole(
                                        

In [14]:
import torch
import transformers

print('torch.cuda.is_available():', torch.cuda.is_available())
print('torch.cuda.device_count():', torch.cuda.device_count())
print('transformers version:', getattr(transformers, '__version__', 'unknown'))


torch.cuda.is_available(): False
torch.cuda.device_count(): 0
transformers version: 5.16.1


# Notes — Authentication, DataLoader, and Training Choices

- **HF authentication**: You logged into Hugging Face. This avoids rate limits, speeds model/dataset downloads, and is required for private repos or pushing models. You can verify with `huggingface-cli whoami` or the helper cell below.

- **Why disable `dataloader_pin_memory` / reduce workers**: This notebook runs on CPU (no CUDA detected). Pinning memory and extra DataLoader workers are optimizations for GPU systems and can increase RAM/worker overhead on CPU-only machines. We set `dataloader_pin_memory=False` and recommend `dataloader_num_workers=0` for stability.

- **Verification first**: Run the small verification cell (below) before training — it checks HF auth, `torch.cuda` availability, and `transformers` version so you don't start long training on the wrong environment.

- **Use small subsets for testing**: When experimenting, use a small slice of the dataset (e.g., 200 examples) to validate the pipeline quickly and avoid OOM issues.

- **If you later run on a GPU**: Re-enable `dataloader_pin_memory=True`, increase `dataloader_num_workers` (e.g., 2–8), and install a CUDA-enabled PyTorch build that matches your drivers.

- **Next steps**:
  1. Run the verification code cell below.
  2. Run a quick end-to-end test on a small dataset subset.
  3. If tests pass and you need speed, move to a GPU environment or cloud instance.


In [12]:
# Quick verification helper
import os

print('HF_TOKEN set in env:', bool(os.environ.get('HF_TOKEN')))
try:
    from huggingface_hub import whoami
    try:
        user = whoami()
        print('huggingface whoami:', user)
    except Exception as e:
        print('whoami check failed:', e)
except Exception as e:
    print('huggingface_hub not installed or whoami failed:', e)

import torch
import transformers
print('torch.cuda.is_available():', torch.cuda.is_available())
print('torch.cuda.device_count():', torch.cuda.device_count())
print('transformers version:', getattr(transformers, '__version__', 'unknown'))


HF_TOKEN set in env: False
[2026-09-03 23:37:32,641: INFO: _client]: HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"]
huggingface whoami: {'type': 'user', 'id': '6a8d58a055aebac28a2e079f', 'name': 'ajcode007', 'fullname': 'santhosh krishna', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1790812800, 'isPro': False, 'avatarUrl': '/avatars/beed4bfa5ac0fad69d717b4bacc1a9e4.svg', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-10-03T17:38:06.000Z'}}
torch.cuda.is_available(): False
torch.cuda.device_count(): 0
transformers version: 5.16.1


In [ ]:
# oauth-ajcode007